# 🎙️ توليد إعلانات بالعامية المصرية — Colab مجاني (GPU T4)

**الخطوات:**
1. من القائمة:  `Runtime` → `Change runtime type` → **T4 GPU** → `Save`
2. شغّل الخلايا بالترتيب (Shift + Enter)
3. عدّل نصوص إعلاناتك في الخلية رقم 4
4. في الآخر هيتنزّلك ملف `ads.zip` فيه كل الأصوات

> الموديل: **VoiceTut-TTS** (مصري أصلي + code-switching، رخصة Apache-2.0 → تجاري مسموح).

In [ ]:
!nvidia-smi

In [ ]:
# تثبيت المكتبات (ممكن ياخد دقيقتين)
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
!pip install -q voicetut-tts soundfile pyloudnorm scipy

In [ ]:
# تحميل الموديل
from voicetut_tts import VoiceTutTTS

MODEL_ID = "mohammedaly22/VoiceTut-TTS"
tts = VoiceTutTTS.from_pretrained(MODEL_ID)
print("✅ الموديل اتحمّل")

In [ ]:
SPEAKER = "Sayed"   # ✏️ غيّره لأي صوت من القائمة
SPEAKERS = ["Abdelrahman","Abdullah","Kamal","Hossam","Mohamed","Omar",
            "Sayed","Zaki","Aly","Essam","Ahmed",
            "Asmaa","Esraa","Hanan","Sarah","Yasmin","Omnia"]
print("الأصوات المتاحة:", ", ".join(SPEAKERS))

# ✏️ عدّل أو زوّد إعلاناتك هنا
ADS = {
    "ad1": "جرب قهوة بون النهاردة! طعم أصلي، ريحة تجنن، وسعر مش هتلاقيه في أي مكان تاني.",
    "ad2": "خصم خمسين بالمية على كل المنتجات النهاردة وبس!",
}

In [ ]:
# توليد كل الإعلانات + معالجة احترافية + تنزيل zip
import os, zipfile
import numpy as np, soundfile as sf, pyloudnorm as pyln
from IPython.display import Audio, display

def master(path, target_lufs=-14.0, ceiling_db=-1.0, multiband=None):
    """معالجة احترافية: لو الريبو متاح بيستخدم سلسلته الكاملة (multiband/EQ)، وإلا نسخة مبسّطة."""
    try:
        from backend.audio.postprocess import postprocess
        postprocess(path, path, target_lufs=target_lufs, true_peak_db=ceiling_db, multiband=multiband)
        return path
    except Exception:
        pass
    data, sr = sf.read(path, dtype="float64")
    try:
        loud = pyln.Meter(sr).integrated_loudness(data)
        if np.isfinite(loud):
            data = data * (10 ** ((target_lufs - loud) / 20))
    except Exception:
        pass
    peak = float(np.max(np.abs(data))) if data.size else 0.0
    ceil = 10 ** (ceiling_db / 20)
    if peak > ceil:
        data = data * (ceil / peak)
    sf.write(path, np.clip(data, -1, 1), sr)
    return path

def to_mp3(path, bitrate=192):
    """تصدير MP3 (backend لو موجود، وإلا نسيب WAV)."""
    try:
        from backend.audio.export import export
        return export(path, "mp3", bitrate=bitrate)
    except Exception:
        return path

os.makedirs("out", exist_ok=True)
paths = []
for name, text in ADS.items():
    out = f"out/{name}.wav"
    tts.synthesize(text, speaker=SPEAKER, output=out,
                   num_step=64, guidance_scale=2.5, speed=0.95)
    master(out, multiband="voice")
    paths.append(out)
    mp3 = to_mp3(out)
    if mp3 != out:
        paths.append(mp3)
    print("✓", out, "(-14 LUFS)")
    display(Audio(out))

with zipfile.ZipFile("ads.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in paths:
        z.write(p, os.path.basename(p))
print("✅ اتجهّز ads.zip")

try:
    from google.colab import files
    files.download("ads.zip")
except Exception as e:
    print("نزّل ads.zip يدوياً من ملفات الجلسة:", e)

## 🧬 (اختياري) استنساخ صوت
ارفع مقطع **10–30 ثانية** بصوت الشخص المطلوب، وبعدين اكتب النص الجديد.

In [ ]:
from google.colab import files
from IPython.display import Audio, display

up = files.upload()          # ارفع ملف wav
ref = list(up.keys())[0]

tts.synthesize(
    "النص اللي عايز تنطقه بصوت ده",
    ref_audio=ref,
    ref_text="",             # لو كتبت نص المقطع بتحسّن الدقة
    output="out/clone.wav",
    num_step=64, guidance_scale=2.5, speed=0.95,
)
display(Audio("out/clone.wav"))
files.download("out/clone.wav")

## 📝 ملاحظات
- لازم تشغّل على **GPU (T4)** وإلا هيكون بطيء جداً.
- Colab مجاني للتوليد، بس **مش** مخصص إنك تسيبه سيرفر شغّال 24 ساعة (مخالف للشروط).
- للسيرفر الدائم مجاناً: استخدم `deploy/modal_app.py`.
- ⚖️ استخدم الأصوات بموافقة أصحابها، وممنوع الانتحال أو التضليل.